In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("TaxiIceberg")

    .config(
        "spark.jars.packages",
        ",".join([
            "org.apache.hadoop:hadoop-aws:3.3.4",
            "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2",
            "com.clickhouse:clickhouse-jdbc:0.6.0"
        ])
    )

    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )

    .config(
        "spark.sql.catalog.demo",
        "org.apache.iceberg.spark.SparkCatalog"
    )

    .config(
        "spark.sql.catalog.demo.type",
        "hadoop"
    )

    .config(
        "spark.sql.catalog.demo.warehouse",
        "s3a://warehouse/iceberg"
    )

    .config(
        "spark.sql.catalog.demo.hadoop.fs.s3a.endpoint",
        "http://minio:9000"
    )

    .config(
        "spark.sql.catalog.demo.hadoop.fs.s3a.access.key",
        "admin"
    )

    .config(
        "spark.sql.catalog.demo.hadoop.fs.s3a.secret.key",
        "password123"
    )

    .config(
        "spark.sql.catalog.demo.hadoop.fs.s3a.path.style.access",
        "true"
    )

    .config(
        "spark.sql.catalog.demo.hadoop.fs.s3a.impl",
        "org.apache.hadoop.fs.s3a.S3AFileSystem"
    )

    .config(
        "spark.sql.catalog.demo.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )

    .getOrCreate()
)

26/05/18 14:48:50 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS demo.taxi")

26/05/18 14:48:56 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


DataFrame[]

In [3]:
df = spark.read.parquet("/data/yellow_tripdata_2023-01.parquet")

In [4]:
df.writeTo("demo.taxi.yellow_trips").create()

26/05/18 14:49:33 WARN S3ABlockOutputStream: Application invoked the Syncable API against stream writing to iceberg/taxi/yellow_trips/data/00005-6-21405861-e2f6-4aae-94bf-e569c60890d1-0-00001.parquet. This is unsupported
                                                                                

In [5]:
spark.sql("""
SELECT count(*)
FROM demo.taxi.yellow_trips
""").show()

+--------+
|count(1)|
+--------+
| 3066766|
+--------+



In [8]:
df_feb = spark.read.parquet("/data/yellow_tripdata_2023-02.parquet")

In [9]:
df_feb.writeTo("demo.taxi.yellow_trips").append()

In [10]:
spark.sql("""
SELECT count(*)
FROM demo.taxi.yellow_trips
""").show()

+--------+
|count(1)|
+--------+
| 5980721|
+--------+



In [7]:
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)



In [8]:
iceberg_df = spark.table("demo.taxi.yellow_trips")

In [9]:
(
    iceberg_df.write
    .format("jdbc")
    .option(
        "url",
        "jdbc:clickhouse://clickhouse:8123/taxi"
    )
    .option(
        "driver",
        "ru.yandex.clickhouse.ClickHouseDriver"
    )
    .option(
        "dbtable",
        "yellow_trips"
    )
    .option(
        "user",
        "default"
    )
    .option(
        "password",
        "password123"
    )
    .mode("append")
    .save()
)

26/05/11 12:38:23 WARN ClickHouseDriver: ******************************************************************************************
26/05/11 12:38:23 WARN ClickHouseDriver: * This driver is DEPRECATED. Please use [com.clickhouse.jdbc.ClickHouseDriver] instead.  *
26/05/11 12:38:23 WARN ClickHouseDriver: * Also everything in package [ru.yandex.clickhouse] will be removed starting from 0.4.0. *
26/05/11 12:38:23 WARN ClickHouseDriver: ******************************************************************************************
26/05/11 12:38:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/05/11 12:38:24 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
                                                                                

In [10]:
df_feb = spark.read.parquet("/data/yellow_tripdata_2023-02.parquet")
df_mar = spark.read.parquet("/data/yellow_tripdata_2023-03.parquet")
df_apr = spark.read.parquet("/data/yellow_tripdata_2023-04.parquet")

In [11]:
spark.sql("""
SELECT *
FROM demo.taxi.yellow_trips.snapshots
""").show(truncate=False)

+-----------------------+-------------------+---------+---------+-----------------------------------------------------------------------------------------------------------------------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id|operation|manifest_list                                                                                                          |summary                                                                                                                                                

In [12]:
df_feb.writeTo("demo.taxi.yellow_trips").append()
df_mar.writeTo("demo.taxi.yellow_trips").append()
df_apr.writeTo("demo.taxi.yellow_trips").append()

In [13]:
spark.sql("""
SELECT *
FROM demo.taxi.yellow_trips.snapshots
""").show(truncate=False)

+-----------------------+-------------------+-------------------+---------+-----------------------------------------------------------------------------------------------------------------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|committed_at           |snapshot_id        |parent_id          |operation|manifest_list                                                                                                          |summary                                                                                                                          

In [16]:
spark.sql("""
SELECT count(*)
FROM demo.taxi.yellow_trips VERSION AS OF 4973049035315457578
""").show()

+--------+
|count(1)|
+--------+
| 9384487|
+--------+



In [17]:
new_data = spark.sql("""
SELECT *
FROM demo.taxi.yellow_trips
WHERE tpep_pickup_datetime >= '2023-02-01'
""")

In [18]:
(
    new_data.write
    .format("jdbc")
    .option(
        "url",
        "jdbc:clickhouse://clickhouse:8123/taxi"
    )
    .option(
        "driver",
        "ru.yandex.clickhouse.ClickHouseDriver"
    )
    .option(
        "dbtable",
        "yellow_trips"
    )
    .option(
        "user",
        "default"
    )
    .option(
        "password",
        "password123"
    )
    .mode("append")
    .save()
)

26/05/11 13:02:26 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/05/11 13:02:26 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/05/11 13:02:26 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
26/05/11 13:02:26 WARN JdbcUtils: Requested isolation level 1, but transactions are unsupported
                                                                                